# Day 4 Advanced Assignment Notebook

This notebook is the advanced-level version of the Day 4 assignment.

We will combine:
- OOP design,
- type hints,
- dataclasses,
- a singleton-like configuration object, and
- a simple analytics pipeline.

In [ ]:
# ------------------------------------------------------------------
# Cell 1: Import libraries and create a pipeline configuration
# ------------------------------------------------------------------
# pandas is used for analytics and summarization.
import pandas as pd

# pathlib helps us handle file paths safely.
from pathlib import Path

# dataclass lets us create lightweight record-like classes.
from dataclasses import dataclass

# Define a singleton-like config class to store shared settings.
class PipelineConfig:
    # Keep one shared instance for the whole pipeline.
    _instance = None

    def __new__(cls):
        # Create the object only once.
        if cls._instance is None:
            cls._instance = super().__new__(cls)
            cls._instance.output_dir = "Day4_Assignment/output"
            cls._instance.encoding = "utf-8"
        # Return the same object every time.
        return cls._instance

config = PipelineConfig()
print("Config object created:", config)
print("Output dir:", config.output_dir)

In [ ]:
# ------------------------------------------------------------------
# Cell 2: Define a dataclass for analytics records
# ------------------------------------------------------------------
# This dataclass stores a patient analytics row with type hints.
@dataclass
class PatientAnalyticsRecord:
    patient_id: int
    condition: str
    department: str
    total_cost: float
    risk_score: int

    # A method that classifies the patient risk level.
    def risk_level(self) -> str:
        if self.risk_score >= 80:
            return "High"
        if self.risk_score >= 50:
            return "Medium"
        return "Low"

record = PatientAnalyticsRecord(1020, "Asthma", "Pulmonology", 1900.25, 45)
print(record)
print("Risk level:", record.risk_level())

In [ ]:
# ------------------------------------------------------------------
# Cell 3: Build the analytics pipeline classes
# ------------------------------------------------------------------
# BaseAnalytics loads the dataset from a file path.
class BaseAnalytics:
    def __init__(self, file_path: str) -> None:
        self.file_path = Path(file_path)

    def load(self) -> pd.DataFrame:
        return pd.read_csv(self.file_path)

# PatientAnalytics inherits from BaseAnalytics and adds business rules.
class PatientAnalytics(BaseAnalytics):
    def add_cost_per_visit(self, data_frame: pd.DataFrame) -> pd.DataFrame:
        working_frame = data_frame.copy()
        working_frame["cost_per_visit"] = working_frame["total_cost"] / working_frame["visits"]
        return working_frame

    def department_summary(self, data_frame: pd.DataFrame) -> pd.DataFrame:
        summary = data_frame.groupby("department").agg(
            patients=("patient_id", "count"),
            avg_cost=("total_cost", "mean"),
            avg_risk=("risk_score", "mean"),
        )
        return summary.sort_values("avg_cost", ascending=False)

    def export_summary(self, summary: pd.DataFrame, filename: str) -> None:
        output_dir = Path(config.output_dir)
        output_dir.mkdir(exist_ok=True)
        output_file = output_dir / filename
        summary.to_csv(output_file)
        print("Advanced report exported to:", output_file)

analytics = PatientAnalytics("Day4_Assignment/data/healthcare_assignment_dataset.csv")
raw_df = analytics.load()
prepared_df = analytics.add_cost_per_visit(raw_df)
summary_df = analytics.department_summary(prepared_df)
print("Department summary:")
print(summary_df.head())

In [ ]:
# ------------------------------------------------------------------
# Cell 4: Export the advanced analytics output
# ------------------------------------------------------------------
# Save the summary table to a CSV file in the output folder.
analytics.export_summary(summary_df, "advanced_department_summary.csv")